In [5]:
from typing import TypedDict
class AgentState(TypedDict):
    message: list[str]
    current_task: str
    final_answer: str

def planner_node(state: AgentState) -> AgentState:
    current_task = state["current_task"]
    plan = f"为任务 '{current_task}' 生成的计划..."

    state["message"].append(plan)
    return state

def executor_node(state: AgentState) -> AgentState:
    latest_plan = state["message"][-1]
    result = f"执行计划 '{latest_plan}' 的结果..."

    state["message"].append(result)
    return state

def should_continue(state: AgentState) -> AgentState:
    if len(state["message"]) < 3:
        return "continue_to_planner"
    else:
        state["final_answer"] = state["message"][-1]
        return "end_workflow"

from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)

workflow.set_entry_point("planner")

workflow.add_edge("planner", "executor")

workflow.add_conditional_edges(
    "executor",
    should_continue,
    {
        "continue_to_planner": "planner",
        "end_workflow": END
    }
)

app = workflow.compile()

inputs = {"current_task": "分析最近的AI行业新闻", "message": []}
for event in app.stream(inputs):
    print(event)

{'planner': {'message': ["为任务 '分析最近的AI行业新闻' 生成的计划..."], 'current_task': '分析最近的AI行业新闻'}}
{'executor': {'message': ["为任务 '分析最近的AI行业新闻' 生成的计划...", "执行计划 '为任务 '分析最近的AI行业新闻' 生成的计划...' 的结果..."], 'current_task': '分析最近的AI行业新闻'}}
{'planner': {'message': ["为任务 '分析最近的AI行业新闻' 生成的计划...", "执行计划 '为任务 '分析最近的AI行业新闻' 生成的计划...' 的结果...", "为任务 '分析最近的AI行业新闻' 生成的计划..."], 'current_task': '分析最近的AI行业新闻'}}
{'executor': {'message': ["为任务 '分析最近的AI行业新闻' 生成的计划...", "执行计划 '为任务 '分析最近的AI行业新闻' 生成的计划...' 的结果...", "为任务 '分析最近的AI行业新闻' 生成的计划...", "执行计划 '为任务 '分析最近的AI行业新闻' 生成的计划...' 的结果..."], 'current_task': '分析最近的AI行业新闻'}}


In [2]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class SearchState(TypedDict):
    messages: Annotated[list, add_messages]
    user_query: str
    search_query: str
    search_result: str
    final_answer: str
    step: str

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from tavily import TavilyClient

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_ID", "deepseek-chat"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0.7
)

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

c:\Users\jd\.conda\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def understand_query_node(state: SearchState)-> dict:
    user_message = state["messages"][-1].content

    understand_prompt = f"""分析用户的查询："{user_message}"
请完成两个任务：
1. 简洁总结用户想要了解什么
2. 生成最适合搜索引擎的关键词（中英文均可，要精准）

格式：
理解：[用户需求总结]
搜索词：[最佳搜索关键词]"""
    
    response = llm.invoke([SystemMessage(content=understand_prompt)])
    response_text = response.content

    search_query = user_message
    if "搜索词：" in response_text:
        search_query = response_text.split("搜索词：")[-1].strip()

    return {
        "user_query": response_text,
        "search_query": search_query,
        "step": "understand",
        "messages": [AIMessage(content=f"我将为你搜索：{search_query}")]
    }

def tavily_search_node(state: SearchState) -> dict:
    search_query = state["search_query"]
    try:
        